In [1]:
!pip install requests

# Documentación Técnica – `analyze_headers.py`


## Propósito del Script

Este script realiza un análisis automatizado de las **cabeceras HTTP de respuesta** de un sitio web. Es utilizado en **fases de reconocimiento** durante auditorías de seguridad para descubrir:

* Tecnologías del servidor
* Plataformas backend
* Políticas de seguridad implementadas (HSTS, cookies)
* Exposición de cabeceras sensibles

El análisis permite detectar malas configuraciones, software expuesto, y ausencia de políticas fundamentales como `Strict-Transport-Security`.


## Librerías utilizadas

```python
import requests
```

* `requests`: librería externa de Python que permite realizar solicitudes HTTP de forma simple y robusta.
* En este contexto, se utiliza para enviar una petición `GET` al servidor objetivo.


## Lógica principal del script

### 1. **Validación y corrección del esquema URL**

```python
if not url.startswith("http://") and not url.startswith("https://"):
    url = "http://" + url
```

* Si el usuario ingresa una URL sin protocolo (`http://` o `https://`), se antepone automáticamente `http://`.
* Esto previene errores de conexión por URLs mal formadas.


### 2. **Petición HTTP y manejo de errores**

```python
try:
    response = requests.get(url)
except requests.exceptions.RequestException as e:
    print(f"Error al realizar la solicitud: {e}")
    return
```

* Se realiza una solicitud `GET` al sitio proporcionado.
* Si hay errores de conexión, DNS, red o timeout, se captura la excepción y se informa al usuario.


### 3. **Validación del código de respuesta**

```python
if response.status_code != 200:
    print(f"Advertencia: el servidor respondió con código {response.status_code}\n")
```

* Si el servidor devuelve un código diferente a 200 (OK), se muestra una advertencia.
* Permite al analista saber si hubo redireccionamientos, errores 4xx o 5xx.


### 4. **Extracción de todas las cabeceras**

```python
headers = response.headers

print("Cabeceras encontradas:\n")
for key, value in headers.items():
    print(f"  {key}: {value}")
```

* Se accede al diccionario de cabeceras devuelto por `requests`.
* Se imprime cada par clave-valor para visualización completa.
* Permite observar detalles adicionales no destacados individualmente.


### 5. **Análisis específico de cabeceras clave**

```python
if "Server" in headers:
    print(f"Server: {headers['Server']}")
else:
    print("Server: No especificado")
```

El mismo patrón se repite para:

* `X-Powered-By`: informa si se usa PHP, ASP.NET, Express, etc.
* `Content-Type`: indica el tipo de contenido que retorna el servidor (HTML, JSON, etc.)
* `Set-Cookie`: muestra cookies entregadas por el servidor (posiblemente inseguras si no usan atributos `HttpOnly`, `Secure`, `SameSite`)
* `Strict-Transport-Security`: indica si el servidor aplica **HSTS**, que obliga al navegador a usar HTTPS exclusivamente.

Cada una de estas cabeceras puede ser utilizada para:

* **Identificar tecnologías** (fingerprinting)
* **Detectar riesgos por exposición de información**
* **Revisar políticas de seguridad aplicadas**


## Entrada y ejecución

```python
if __name__ == "__main__":
    print("Herramienta de Análisis de Cabeceras HTTP")
    target_url = input("Ingresa la URL del sitio a analizar: ")
    analyze_headers(target_url)
```

* Al ejecutarse directamente, el script solicita al usuario una URL.
* Llama a la función principal `analyze_headers(...)` con la URL proporcionada.
* Toda la salida se imprime directamente en consola.


## Importancia en Ciberseguridad

Este script simula lo que hacen herramientas como **WhatWeb**, **Wappalyzer**, o extensiones de **Burp Suite**. Ayuda a responder preguntas como:

* ¿Qué software usa el servidor?
* ¿Tiene habilitada protección HTTPS?
* ¿Expone cookies sin atributos de seguridad?
* ¿Divulga su tecnología backend?

En entornos reales, estas cabeceras permiten a un atacante identificar vectores de ataque o tecnologías con vulnerabilidades conocidas.



In [ ]:
import requests

def analyze_headers(url):
    if not url.startswith("http://") and not url.startswith("https://"):
        url = "http://" + url

    print(f"\nAnalizando cabeceras de: {url}\n")

    try:
        response = requests.get(url)
    except requests.exceptions.RequestException as e:
        print(f"Error al realizar la solicitud: {e}")
        return

    if response.status_code != 200:
        print(f"Advertencia: el servidor respondió con código {response.status_code}\n")

    headers = response.headers

    print("Cabeceras encontradas:\n")
    for key, value in headers.items():
        print(f"  {key}: {value}")

    print("\nAnálisis específico:\n")

    if "Server" in headers:
        print(f"Server: {headers['Server']}")
    else:
        print("Server: No especificado")

    if "X-Powered-By" in headers:
        print(f"X-Powered-By: {headers['X-Powered-By']}")
    else:
        print("X-Powered-By: No especificado")

    if "Content-Type" in headers:
        print(f"Content-Type: {headers['Content-Type']}")
    else:
        print("Content-Type: No especificado")

    if "Set-Cookie" in headers:
        print(f"Set-Cookie: {headers['Set-Cookie']}")
    else:
        print("Set-Cookie: No especificado")

    if "Strict-Transport-Security" in headers:
        print(f"Strict-Transport-Security: {headers['Strict-Transport-Security']}")
    else:
        print("Strict-Transport-Security: No especificado")

if __name__ == "__main__":
    print("Herramienta de Análisis de Cabeceras HTTP")
    target_url = input("Ingresa la URL del sitio a analizar: ")
    analyze_headers(target_url)
